# ARCONT — Generador 3D gratuito
Genera un `.glb` desde una imagen usando **Stable Fast 3D** (principal) o **TripoSR** (respaldo).

**Recomendado:** `sf3d`. Produce UV/materiales y admite remallado. El modelo es gratuito pero requiere aceptar su acceso en Hugging Face una vez.

En Colab: **Entorno de ejecución → Cambiar tipo de entorno → GPU**. Después ejecuta las celdas en orden.

In [ ]:
import os, subprocess, sys, shutil, glob
try:
    import torch
    print('CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1))
except Exception as e:
    print(e)
print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null'))

In [ ]:
# CONFIGURACIÓN
BACKEND = 'sf3d'      # 'sf3d' o 'triposr'
TARGET_VERTICES = 2500
TEXTURE_RESOLUTION = 1024
OUTPUT_NAME = 'arcont_character.glb'
print('Backend:', BACKEND)

## Stable Fast 3D: acceso gratuito
Solo si usas `sf3d`: acepta el modelo en https://huggingface.co/stabilityai/stable-fast-3d y crea un token **Read** en Hugging Face. El token se introduce de forma oculta en Colab; no se guarda en GitHub.

In [ ]:
from getpass import getpass
HF_TOKEN = ''
if BACKEND == 'sf3d':
    HF_TOKEN = getpass('Hugging Face token (Read): ')
    if not HF_TOKEN:
        raise ValueError('Stable Fast 3D necesita un token gratuito de Hugging Face.')

In [ ]:
# INSTALACIÓN AUTOMÁTICA
os.chdir('/content')
if BACKEND == 'sf3d':
    subprocess.run('rm -rf stable-fast-3d', shell=True)
    subprocess.run('git clone --depth 1 https://github.com/Stability-AI/stable-fast-3d.git', shell=True, check=True)
    os.chdir('/content/stable-fast-3d')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'setuptools==69.5.1', 'wheel', 'huggingface_hub'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    os.chdir('/content')
    subprocess.run('rm -rf TripoSR', shell=True)
    subprocess.run('git clone --depth 1 https://github.com/VAST-AI-Research/TripoSR.git', shell=True, check=True)
    os.chdir('/content/TripoSR')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime'], check=True)
print('Instalación terminada.')

In [ ]:
# SUBE LA IMAGEN DEL PERSONAJE
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise ValueError('No se subió ninguna imagen.')
input_name = next(iter(uploaded))
input_path = '/content/arcont_input' + os.path.splitext(input_name)[1].lower()
with open(input_path, 'wb') as f:
    f.write(uploaded[input_name])
print('Entrada:', input_path)

In [ ]:
# GENERACIÓN
outdir = '/content/arcont_output'
shutil.rmtree(outdir, ignore_errors=True)
os.makedirs(outdir, exist_ok=True)
if BACKEND == 'sf3d':
    os.chdir('/content/stable-fast-3d')
    cmd = [sys.executable, 'run.py', input_path, '--output-dir', outdir, '--texture-resolution', str(TEXTURE_RESOLUTION), '--remesh_option', 'quad', '--target_vertex_count', str(TARGET_VERTICES)]
    print('Generando con Stable Fast 3D...')
    result = subprocess.run(cmd)
    if result.returncode != 0:
        print('El remallado quad falló; reintentando sin remallado para no perder la generación...')
        shutil.rmtree(outdir, ignore_errors=True); os.makedirs(outdir, exist_ok=True)
        cmd = [sys.executable, 'run.py', input_path, '--output-dir', outdir, '--texture-resolution', str(TEXTURE_RESOLUTION), '--remesh_option', 'none']
        subprocess.run(cmd, check=True)
else:
    os.chdir('/content/TripoSR')
    cmd = [sys.executable, 'run.py', input_path, '--device', 'cuda' if __import__('torch').cuda.is_available() else 'cpu', '--chunk-size', '8192', '--mc-resolution', '256', '--model-save-format', 'glb', '--output-dir', outdir]
    print('Generando con TripoSR...')
    subprocess.run(cmd, check=True)
print('Generación terminada.')

In [ ]:
# ENCUENTRA Y DESCARGA EL GLB
matches = glob.glob(outdir + '/**/*.glb', recursive=True)
if not matches:
    raise FileNotFoundError('No se encontró ningún GLB en la salida.')
src = matches[0]
final_path = '/content/' + OUTPUT_NAME
shutil.copy2(src, final_path)
print('Resultado:', final_path, '|', round(os.path.getsize(final_path)/1024/1024, 2), 'MB')
files.download(final_path)

## Para Arcont
Abre el `.glb` descargado en Nomad Sculpt para revisarlo. Si la geometría sale bien, el siguiente paso será automatizar limpieza, retopología final, módulos de gore y rig para Godot.